This notebook shows how to:
1. load a sequence of .tif files (that correspond to a continuous imaging session)
2. motion correct save the results out as a single .hdf5 file (which contains the shifts and motion corrected imaging stack)
3. Load the reults into a fast viewer

In [1]:
import fastplotlib as fpl
import tifffile
import masknmf
import os
import numpy as np
import torch

import matplotlib.pyplot as plt
%load_ext autoreload

Unable to find extension: VK_EXT_acquire_drm_display
Unable to find extension: VK_EXT_physical_device_drm
No windowing system present. Using surfaceless platform
No config found!
No config found!
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA TITAN RTX,DiscreteGPU,Vulkan,555.42.02
❗ limited,"llvmpipe (LLVM 12.0.0, 256 bits)",CPU,Vulkan,Mesa 21.2.6 (LLVM 12.0.0)
❌,NVIDIA TITAN RTX/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 555.42.02


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


In [2]:
## Note: write out a single memmappable tiff file for the fastest processing (i/o is the bottleneck here)
## A sequence of tiff files is significantly slower than a single, memmap-compatible file

# Use this if you have a sequence of tiff files
files = ["/data/lab/NectowLab/Mouse7_VRE8F1/rTigre_VRE8F1_PBG_at60_Capsaicin_at360_20Hz_20msec_part1.tif",
         "/data/lab/NectowLab/Mouse7_VRE8F1/rTigre_VRE8F1_PBG_at60_Capsaicin_at360_20Hz_20msec_part2.tif"]
raw_data = masknmf.TiffSeriesLoader(files, memmap = False)

# Use this if you have a single tiff file (desirable, with memmap = True)
# raw_data = masknmf.TiffArray(file, memmap = False) 

In [7]:
hp_filter_kernel = masknmf.motion_correction.spatial_filters.compute_highpass_filter_kernel([4, 4]).cuda()

hp_filter_func = lambda x:torch.nn.functional.relu(masknmf.motion_correction.spatial_filters.image_filter(x.to(hp_filter_kernel.device), hp_filter_kernel))
filt_arr = masknmf.motion_correction.FilteredArray(raw_data,
                                                   hp_filter_func,
                                                   device = "cuda")

In [7]:
max_shifts = [30, 30]

"""
This is how you'd run rigid motion correction
Uncomment + run this code to run rigid registration
"""
# moco_strat = masknmf.RigidMotionCorrector(max_shifts,
#                                          batch_size = 200,
#                                          pixel_weighting = None)


"""
Piecewise rigid motion correction explanation
num_blocks = we break up the FOV into this many spatial blocks (roughly same size) to estimate local motion (on top of rigid motion). 

overlaps: the number of pixels by which these blocks overlap (roughly)

max_rigid_shifts: this code runs rigid registration BEFORE piecewise rigid, this is the max shifts for the rigid step

max_deviation_rigid: on top of the rigid shift, this param specifies how much each block's local motion can deviate from the global rigid shift

Things to look out for: 
1. If you see that rigid shifts worked but there is warping, you can reduce max_deviation_rigid
2. 

"""
## This is how you run the rigid + piecewise rigid pipeline
## num_blocks splits the FOV into blocks of size height / num_blocks[0], width / num_blocks[1]. You want block size to be roughly 100 x 100
moco_strat = masknmf.PiecewiseRigidMotionCorrector(num_blocks = [5, 5], #each block has roughly height / num_blocks[0] by width / num_blocks[1]
                                                   overlaps = [30, 30],
                                                   max_rigid_shifts = [30, 30],
                                                   max_deviation_rigid = [3, 3])
moco_strat.compute_template(filt_arr)

In [1]:
moco_results = masknmf.RegistrationArray(filt_arr,
                                         moco_strat,
                                        target_movie=raw_data)

moco_results.export("./moco_results.hdf5")

In [3]:
## Load the moco results from disk now:
moco_results = masknmf.RegistrationArray.from_hdf5("./moco_results.hdf5")

In [8]:
moco_vis = masknmf.MotionCorrectionVis(raw_data, moco_results)

In [6]:
moco_vis.show()